In [0]:
import dlt
import pyspark.sql.functions as F

In [0]:
# Step 1: Bronze
@dlt.table(name="bronze_prospects",
    comment="Raw prospect data from FanGraphs API (bronze layer)",
    table_properties={
    "quality": "bronze"
  }
)
def bronze_prospects():
    # Input Parameters
    season = spark.conf.get("season", "2025")
    board_type = spark.conf.get("board_type", "prospect")
    path = f"/Volumes/alexander_booth/rag_demo/prospect_data/source=fangraphs/board_type={board_type}/season={season}"
    
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("multiLine", "true")
            .load(path)
            .withColumn("load_date", F.current_timestamp()) 
            .withColumn("source_file", F.col("_metadata.file_path"))
            .withColumn("file_mod_time", F.col("_metadata.file_modification_time"))
            .withColumn("file_size", F.col("_metadata.file_size"))
    )

In [0]:
# Step 2: Enriched view
@dlt.view
@dlt.expect_or_drop("valid_minorMasterId", "minorMasterId IS NOT NULL")
def enriched_prospects():
    raw = dlt.read_stream("bronze_prospects")

    # Text Blocks
    tldr_block = F.when(
      F.col("TLDR").isNotNull(), F.concat(F.lit("TLDR:\n"), F.col("TLDR"))
    )

    summary_block = F.when(
        F.col("Summary").isNotNull(), F.concat(F.lit("Full Report:\n"), F.col("Summary"))
    )

    ovr_summary_block = F.when(
        F.col("Summary").isNotNull() &
        F.col("Ovr_Summary").isNotNull() &
        (F.col("Summary") != F.col("Ovr_Summary")),
        F.col("Ovr_Summary")
    ).when(
        F.col("Summary").isNull() & F.col("Ovr_Summary").isNotNull(),
        F.concat(F.lit("Full Report:\n"), F.col("Ovr_Summary"))
    )

    return (
    raw.withColumn("primary_key", F.concat_ws("_", F.col("minorMasterId"), F.col("Season")))
       .withColumn("combined_text", F.concat_ws("\n\n", tldr_block, summary_block, ovr_summary_block))
       .drop(
           "_rescued_data",
           "file_mod_time",
           "file_size"
       )
    )


# Step 3: Register the target table
dlt.create_target_table(
    name="silver_prospects",
    comment="Table storing silver layer scouting reports for prospects",
    table_properties={
        "delta.enableChangeDataFeed": "true",
        "quality": "silver"
    }
)

# Step 4: Apply merge logic to that table
dlt.apply_changes(
    target="silver_prospects",
    source="enriched_prospects",
    keys=["primary_key"],
    sequence_by=F.col("ingest_date"),
    stored_as_scd_type=1
)

In [0]:
# Step 5: Narrow view from silver_prospects for index
@dlt.table
def index_input_changes():
    return dlt.read("silver_prospects").select(
        "primary_key", "Season", "minorMasterId", "playerName", "combined_text", "ingest_date"
    )

# Step 6: Register physical Delta table for vector index input
dlt.create_target_table(
    name="silver_prospects_index_input",
    comment="Slimmed down version of silver_prospects used for embedding and vector indexing",
    table_properties={
        "delta.enableChangeDataFeed": "true",
        "quality": "silver"
    }
)

# Step 7: Apply merge logic to populate it
dlt.apply_changes(
    target="silver_prospects_index_input",
    source="index_input_changes",
    keys=["primary_key"],
    sequence_by=F.col("ingest_date"),
    stored_as_scd_type=1
)